# Diabetes Disease Detection: XGBoost vs. PyTorch Neural Network

This notebook provides an interactive walkthrough of the diabetes disease detection pipeline. We use the **Pima Indians Diabetes Dataset** to train and compare two different model paradigms:
1. **XGBoost Classifier**: An optimized gradient boosting framework for tabular data.
2. **Artificial Neural Network (ANN)**: A feedforward neural network built using PyTorch.

### Pipeline Overview
- **Exploratory Data Analysis (EDA)**: Visualizing correlations and feature distributions.
- **Preprocessing**: Downloading, splitting (Train/Val/Test), imputing zero values (missing measurements) without data leakage, and scaling features.
- **XGBoost Training**: Hyperparameter tuning via GridSearchCV (5-fold CV) and plotting feature importances.
- **Neural Network Training**: Custom PyTorch model with Batch Normalization, Dropout, and Early Stopping.
- **Evaluation**: Comparing model metrics (Accuracy, Precision, Recall, F1-Score, ROC-AUC) and plotting ROC curves & Confusion Matrices.

## 1. Environment Setup & Data Loading

Let's import necessary libraries and append the `src/` directory to the path so we can import our modules.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src/ to system path
sys.path.append(os.path.abspath("../src"))

from utils import RAW_DATA_PATH, set_seed
import data_preprocessing as dp
import train_xgboost as tx
import train_ann as ta
import evaluate as ev

# Set seed for reproducibility
set_seed(42)
sns.set_theme(style="whitegrid")

### Download & Load the Dataset

In [ ]:
# Download if not present
dp.download_dataset()

# Load raw data for EDA
df_raw = pd.read_csv(RAW_DATA_PATH)
df_raw.head()

## 2. Exploratory Data Analysis (EDA)

Let's explore the raw features and verify class distributions.

In [ ]:
print("=== Dataset Info ===")
df_raw.info()
print("\n=== Class Balance ===")
print(df_raw["Outcome"].value_counts(normalize=True))

### Feature Correlations
Let's visualize how features correlate with each other and the outcome.

In [ ]:
plt.figure(figsize=(10, 8))
corr = df_raw.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", square=True, linewidths=0.5)
plt.title("Correlation Matrix (Raw Features)", fontsize=14, fontweight="bold")
plt.show()

### Note on Zeros representing Missing Data
Many columns have 0 values which represent missing data rather than actual measurements. Let's count them.

In [ ]:
cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
for c in cols:
    zeros = (df_raw[c] == 0).sum()
    pct = (df_raw[c] == 0).mean() * 100
    print(f"{c}: {zeros} zero values ({pct:.2f}%)")

## 3. Preprocessing & Splitting

We split the data first, and then impute the missing values on each split using the **training median** to prevent data leakage. We scale the features using `StandardScaler` fitted on the training split.

In [ ]:
train_df, val_df, test_df = dp.preprocess_and_split()
print(f"Preprocessed Train shape: {train_df.shape}")
print(f"Preprocessed Val shape: {val_df.shape}")
print(f"Preprocessed Test shape: {test_df.shape}")

## 4. Model Training

### A. XGBoost Classifier

We run hyperparameter tuning with 5-fold cross-validation on the combined train+val data.

In [ ]:
tx.train_xgboost()

Let's display the saved feature importance plot.

In [ ]:
from PIL import Image
img_fi = Image.open("../images/xgboost_feature_importance.png")
plt.figure(figsize=(10, 6))
plt.imshow(img_fi)
plt.axis('off')
plt.show()

### B. PyTorch Artificial Neural Network

We train a feedforward neural network in PyTorch using binary cross-entropy loss, dropout regularization, and early stopping.

In [ ]:
ta.train_ann()

Let's inspect the training/validation loss curve.

In [ ]:
img_loss = Image.open("../images/ann_loss_curve.png")
plt.figure(figsize=(10, 6))
plt.imshow(img_loss)
plt.axis('off')
plt.show()

## 5. Comparative Evaluation

We evaluate both models on the unseen holdout test set.

In [ ]:
ev.evaluate_models()

### Visual Comparisons
Let's look at the ROC Curves and Confusion Matrices.

In [ ]:
img_roc = Image.open("../images/roc_curve_comparison.png")
img_cm = Image.open("../images/confusion_matrices_comparison.png")

fig, axes = plt.subplots(2, 1, figsize=(12, 16))
axes[0].imshow(img_roc)
axes[0].axis('off')
axes[0].set_title("ROC Curves", fontsize=14, fontweight="bold")

axes[1].imshow(img_cm)
axes[1].axis('off')
axes[1].set_title("Confusion Matrices", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

## Conclusion & Findings

- **Tabular Advantage:** XGBoost often performs slightly better or comparable on small structured tabular datasets like this because boosting methods excel at building step-function boundaries over numeric fields.
- **Deep Learning Scaling:** PyTorch Neural Networks are highly customizable and can scale to very large datasets, but require careful tuning (Dropout, Batch Normalization, Weight Decay) to avoid overfitting on smaller datasets (768 rows).
- **Metric Choice:** In a clinical context, a False Negative (missing a diabetic patient) is more dangerous than a False Positive (additional testing). Thus, comparing models based on **Recall (Sensitivity)** or balancing via F1-Score/AUC is critical.